In [1]:
import os

os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "python")
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION", "3")

import gc
import random
import numpy as np
import pandas as pd

try:
    import cv2
except Exception:
    cv2 = None

try:
    import pydicom as dicom
except Exception:
    dicom = None

from sklearn.model_selection import KFold, train_test_split



In [2]:
MIN_MAX = {
    "Weeks": (-5.0, 133.0),
    "FVC": (827.0, 6399.0),
    "Percent": (28.877577, 153.145378),
    "Age": (49.0, 88.0),
    "typical_fvc": (827.0, 6399.0),
}



In [3]:
IMG_SIZE = 256
NUM_OF_SCANS = 12
BATCH_SIZE = 32

BASE_INPUT = "../input/osic-pulmonary-fibrosis-progression"
if not os.path.exists(BASE_INPUT):
    BASE_INPUT = "/kaggle/input/osic-pulmonary-fibrosis-progression"
if not os.path.exists(BASE_INPUT):
    BASE_INPUT = "/kaggle/data/osic-pulmonary-fibrosis-progression"

TEST_DF = pd.read_csv(os.path.join(BASE_INPUT, "test.csv"))
TRAIN_DF = pd.read_csv(os.path.join(BASE_INPUT, "train.csv"))
SAMPLE_SUBMISSION = pd.read_csv(os.path.join(BASE_INPUT, "sample_submission.csv"))

model_path = "../input/osic-resnet-v5"
model_weights = []
if os.path.isdir(model_path):
    model_weights = [os.path.join(model_path, x) for x in os.listdir(model_path)]

training_features = [
    "Weeks",
    "min_week",
    "min_week_FVC",
    "typical_fvc",
    "Age",
    "Male",
    "Female",
    "Never smoked",
    "Currently smokes",
    "Ex-smoker",
]
num_of_features = len(training_features)




In [4]:
def create_typical_fvc(df):
    df = df.copy()
    df["typical_fvc"] = df["FVC"] / df["Percent"] * 100.0
    return df


def normalize(df):
    df = df.copy()
    for feature, min_max in MIN_MAX.items():
        if feature in df.columns:
            df[feature] = (df[feature] - min_max[0]) / (min_max[1] - min_max[0])
    return df


def add_onehots(df):
    df = df.copy()
    for col in ["Never smoked", "Currently smokes", "Ex-smoker"]:
        if col not in df.columns:
            df[col] = 0
    for col in ["Male", "Female"]:
        if col not in df.columns:
            df[col] = 0

    df["Never smoked"] = (df["SmokingStatus"] == "Never smoked").astype("uint8")
    df["Currently smokes"] = (df["SmokingStatus"] == "Currently smokes").astype("uint8")
    df["Ex-smoker"] = (df["SmokingStatus"] == "Ex-smoker").astype("uint8")
    df["Male"] = (df["Sex"] == "Male").astype("uint8")
    df["Female"] = (df["Sex"] == "Female").astype("uint8")
    return df


TEST_DF = create_typical_fvc(TEST_DF)
TEST_DF["min_week"] = np.zeros(len(TEST_DF), dtype="float32")
TEST_DF["min_week_FVC"] = np.zeros(len(TEST_DF), dtype="float32")
TEST_DF = add_onehots(TEST_DF)
TEST_DF = normalize(TEST_DF)

for patient in np.unique(TEST_DF["Patient"]):
    m = TEST_DF["Patient"] == patient
    TEST_DF.loc[m, "min_week"] = TEST_DF.loc[m, "Weeks"].min()
    TEST_DF.loc[m, "min_week_FVC"] = TEST_DF.loc[m, "FVC"].values[0]

TEST_DF.head()



/tmp/ipykernel_11/840429273.py:40: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.036231884057971016' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  TEST_DF.loc[m, "min_week"] = TEST_DF.loc[m, "Weeks"].min()
/tmp/ipykernel_11/840429273.py:41: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.5348169418521177' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  TEST_DF.loc[m, "min_week_FVC"] = TEST_DF.loc[m, "FVC"].values[0]


,Patient,Weeks,FVC,Percent,Age,Sex,SmokingStatus,typical_fvc,min_week,min_week_FVC,Never smoked,Currently smokes,Ex-smoker,Male,Female
0,ID00014637202177757139317,0.036232,0.534817,0.492477,0.179487,Male,Ex-smoker,0.610086,0.036232,0.534817,0,0,1,1,0
1,ID00019637202178323708467,0.130435,0.228464,0.514865,0.871795,Female,Ex-smoker,0.257448,0.130435,0.228464,0,0,1,0,1
2,ID00047637202184938901501,0.050725,0.446159,0.491293,0.487179,Male,Ex-smoker,0.512742,0.050725,0.446159,0,0,1,1,0
3,ID00082637202201836229724,0.173913,0.375269,0.570681,0.000000,Female,Currently smokes,0.376346,0.173913,0.375269,0,1,0,0,1
4,ID00126637202218610655908,0.166667,0.277818,0.245417,0.743590,Male,Ex-smoker,0.569454,0.166667,0.277818,0,0,1,1,0


In [5]:
volumes = {}
if len(model_weights) > 0 and (cv2 is not None) and (dicom is not None):

    def get_pixels_hu(scan):
        image = scan.pixel_array
        image = image.astype(np.int16)

        slope = float(getattr(scan, "RescaleSlope", 1.0))
        intercept = float(getattr(scan, "RescaleIntercept", 0.0))
        window_center = -200
        window_width = 2000
        if slope != 1:
            image = slope * image.astype(np.float64)
            image = image.astype(np.int16)
        image += np.int16(intercept)

        image_min = window_center - window_width // 2
        image_max = window_center + window_width // 2
        image[image < image_min] = image_min
        image[image > image_max] = image_max

        image = image.astype(np.float64)
        image = (image - image_min) / (image_max - image_min) * 255.0
        return image.astype(np.uint8)

    test_dicom_root = os.path.join(BASE_INPUT, "test")
    for patient_id in np.unique(TEST_DF["Patient"]):
        image_folder = os.path.join(test_dicom_root, patient_id)
        if not os.path.isdir(image_folder):
            continue
        image_files = sorted(
            os.listdir(image_folder),
            key=lambda x: (
                int(os.path.splitext(x)[0]) if os.path.splitext(x)[0].isdigit() else x
            ),
        )
        if len(image_files) == 0:
            continue
        mid = len(image_files) // 2
        start = max(0, mid - NUM_OF_SCANS // 2)
        end = min(len(image_files), start + NUM_OF_SCANS)
        image_files = image_files[start:end]

        scans = [
            dicom.dcmread(os.path.join(image_folder, image_file))
            for image_file in image_files
        ]
        images = np.asarray(
            [cv2.resize(get_pixels_hu(scan), (IMG_SIZE, IMG_SIZE)) for scan in scans],
            dtype="float32",
        )
        if images.shape[0] < NUM_OF_SCANS:
            pad_n = NUM_OF_SCANS - images.shape[0]
            images = np.concatenate(
                [images, np.repeat(images[-1:, ...], pad_n, axis=0)], axis=0
            )
        volumes[patient_id] = images

len(volumes)



0

In [6]:
if len(model_weights) > 0:
    import tensorflow as tf
    from tensorflow.keras import backend as K
    from tensorflow.keras.layers import (
        Input,
        Conv3D,
        BatchNormalization,
        Activation,
        MaxPool3D,
        Add,
        GlobalAveragePooling3D,
        Concatenate,
        Dense,
    )
    from tensorflow.keras.models import Model
    from tensorflow.keras.utils import Sequence

    class Dataset(Sequence):
        def __init__(self, batch_size=BATCH_SIZE, mode=0):
            self.indices = np.arange(0, len(SAMPLE_SUBMISSION), 1)
            self.batch_size = batch_size
            self.mode = mode  # 0 - Training, 1 - Test

        def __len__(self):
            return int(np.ceil(len(self.indices) / self.batch_size))

        def get_tabular(self, patient, week):
            week = (float(week) - MIN_MAX["Weeks"][0]) / (
                MIN_MAX["Weeks"][1] - MIN_MAX["Weeks"][0]
            )
            tabular = [week]
            tabular += list(
                TEST_DF[training_features[1:]][TEST_DF["Patient"] == patient].values[0]
            )
            return np.asarray(tabular, dtype="float32")

        def get_volume(self, patient_id):
            if patient_id in volumes:
                return volumes[patient_id]
            return np.zeros((NUM_OF_SCANS, IMG_SIZE, IMG_SIZE), dtype=np.float32)

        def __getitem__(self, index):
            start = index * self.batch_size
            end = min(len(self.indices), (index + 1) * self.batch_size)
            indices = self.indices[start:end]

            Patient_Week = np.asarray(SAMPLE_SUBMISSION["Patient_Week"][indices])
            patient = [x.split("_")[0] for x in Patient_Week]
            week = [x.split("_")[1] for x in Patient_Week]
            images = (
                np.asarray(
                    [self.get_volume(patient_id) for patient_id in patient],
                    dtype=np.float32,
                )
                / 255.0
            )
            images = np.expand_dims(images, axis=4)
            tabulars = np.asarray(
                [self.get_tabular(patient[i], week[i]) for i in range(len(patient))],
                dtype="float32",
            )
            return [images, tabulars]

    def swish(x):
        return x * K.sigmoid(x)

    def conv_block(x, num_of_filters):
        x = Conv3D(
            num_of_filters,
            kernel_size=(3, 1, 1),
            padding="same",
            kernel_initializer="he_uniform",
        )(x)
        x = BatchNormalization()(x)
        x = Activation("relu")(x)
        x = Conv3D(
            num_of_filters,
            kernel_size=(1, 3, 3),
            padding="same",
            kernel_initializer="he_uniform",
        )(x)
        x = BatchNormalization()(x)
        x = Activation("relu")(x)
        x = Conv3D(
            num_of_filters,
            kernel_size=(1, 1, 1),
            padding="same",
            kernel_initializer="he_uniform",
        )(x)
        x = BatchNormalization()(x)
        x = Activation("relu")(x)
        return x

    def residual_block(x, num_of_filters):
        x1 = conv_block(x, num_of_filters)
        x2 = conv_block(x1, num_of_filters)
        return Add()([x1, x2])

    def build_3d_resnet(input_tensor):
        c1 = Conv3D(64, kernel_size=(5, 7, 7), strides=(1, 2, 2), padding="same")(
            input_tensor
        )
        b1 = BatchNormalization()(c1)
        a1 = Activation("relu")(b1)
        p1 = MaxPool3D(pool_size=(3, 3, 3), strides=(1, 2, 2))(a1)

        r1 = residual_block(p1, 64)
        r1 = residual_block(r1, 64)
        p1 = MaxPool3D(pool_size=(3, 3, 3), strides=(1, 2, 2))(r1)

        r2 = residual_block(p1, 128)
        r2 = residual_block(r2, 128)
        p2 = MaxPool3D(pool_size=(3, 3, 3), strides=(1, 2, 2))(r2)

        r3 = residual_block(p2, 256)
        r3 = residual_block(r3, 256)
        return r3

    def build_model(weights):
        input_img = Input(shape=(NUM_OF_SCANS, IMG_SIZE, IMG_SIZE, 1))
        x = build_3d_resnet(input_img)
        input_latent = GlobalAveragePooling3D()(x)

        input_tabular = Input(shape=(num_of_features,))
        x = Concatenate()([input_latent, input_tabular])
        x = Dense(100)(x)
        x = Dense(100)(x)
        out = Dense(3)(x)

        model = Model(
            inputs=[input_img, input_tabular], outputs=out, name="tabular_model"
        )
        model.load_weights(weights)
        return model




In [7]:
models = []
if len(model_weights) > 0:
    models = [build_model(weights) for weights in model_weights]

len(models), (len(model_weights) > 0)



(0, False)

In [8]:
if len(models) > 0:
    test_gen = Dataset(mode=1)
    predictions = np.zeros((len(SAMPLE_SUBMISSION), 3), dtype="float32")
    for model in models:
        predictions += model.predict(test_gen, verbose=1)
else:
    # Change (score-improving, metric-aligned, same core logic):
    # Keep linear-in-weeks prediction, but learn a GLOBAL linear FVC model using all train rows
    # and then anchor each patient to their observed baseline (test has one row/patient).
    # This uses available clinical covariates more directly than "slope-only" and typically improves FVC accuracy.
    TRAIN_DF2 = TRAIN_DF.copy()
    TRAIN_DF2 = create_typical_fvc(TRAIN_DF2)
    TRAIN_DF2 = add_onehots(TRAIN_DF2)

    # Compute per-patient baseline week/FVC for anchoring and delta modeling
    train_group = TRAIN_DF2.groupby("Patient", sort=False)
    patient_baseline = {}
    for pid, g in train_group:
        g = g.sort_values("Weeks")
        w = g["Weeks"].values.astype(np.float64)
        y = g["FVC"].values.astype(np.float64)
        if len(w) == 0:
            continue
        idx0 = int(np.argmin(np.abs(w - 0.0)))
        w0 = float(w[idx0])
        y0 = float(y[idx0])
        patient_baseline[pid] = (w0, y0)

    # Features for the global linear model (intercept is added later)
    feat_cols = [
        "Weeks",
        "typical_fvc",
        "Age",
        "Male",
        "Female",
        "Never smoked",
        "Currently smokes",
        "Ex-smoker",
    ]

    # Build a ridge regression for predicting delta-FVC relative to each patient's baseline:
    # dy = FVC - FVC_baseline ≈ beta @ [dWeeks, typical_fvc, Age, onehots...]
    # Change rationale: removes patient-level offset (huge variance) and focuses model on progression,
    # which usually improves both FVC and confidence calibration.
    rows_X = []
    rows_y = []
    for pid, g in train_group:
        if pid not in patient_baseline:
            continue
        w0, y0 = patient_baseline[pid]
        g = g.sort_values("Weeks")
        for _, r in g.iterrows():
            dw = float(r["Weeks"]) - w0
            x = [
                dw,
                float(r["typical_fvc"]),
                float(r["Age"]),
                float(r["Male"]),
                float(r["Female"]),
                float(r["Never smoked"]),
                float(r["Currently smokes"]),
                float(r["Ex-smoker"]),
            ]
            rows_X.append(x)
            rows_y.append(float(r["FVC"]) - y0)

    X = np.asarray(rows_X, dtype=np.float64)
    y = np.asarray(rows_y, dtype=np.float64)

    # Safe global sigma fallback
    global_fvc_std = float(TRAIN_DF2["FVC"].std(ddof=0)) if len(TRAIN_DF2) else 300.0
    global_fvc_std = float(np.clip(global_fvc_std, 70.0, 1000.0))

    if X.shape[0] >= 10:
        X1 = np.concatenate([np.ones((X.shape[0], 1), dtype=np.float64), X], axis=1)
        lam = 50.0
        A = X1.T @ X1 + lam * np.eye(X1.shape[1], dtype=np.float64)
        beta = np.linalg.solve(A, X1.T @ y)
        yhat = X1 @ beta
        resid = y - yhat
        mae_global = float(np.mean(np.abs(resid))) if resid.size else 0.0
        sigma_global = (
            float(np.sqrt(2.0) * mae_global) if mae_global > 0 else global_fvc_std
        )
        sigma_global = float(np.clip(sigma_global, 70.0, 1000.0))
    else:
        beta = None
        sigma_global = global_fvc_std

    # Build per-patient residual scale (shrunk to global) for Confidence
    patient_sigma = {}
    patient_n = {}
    if beta is not None:
        # Precompute per-patient residual MAE under the global delta model
        resid_by_pid = {}
        count_by_pid = {}
        X1 = np.concatenate([np.ones((X.shape[0], 1), dtype=np.float64), X], axis=1)
        yhat = X1 @ beta
        abs_resid = np.abs(y - yhat)
        # Re-loop to assign residuals to patients in the same order used above
        k = 0
        for pid, g in train_group:
            if pid not in patient_baseline:
                continue
            nrows = len(g)
            if nrows <= 0:
                continue
            r = abs_resid[k : k + nrows]
            k += nrows
            resid_by_pid[pid] = (
                float(np.mean(r)) if r.size else float(sigma_global / np.sqrt(2.0))
            )
            count_by_pid[pid] = int(nrows)

        for pid in resid_by_pid:
            mae = float(resid_by_pid[pid])
            sigma = float(np.sqrt(2.0) * mae) if mae > 0 else sigma_global
            # Change (metric-aligned): shrink per-patient sigma to global based on sample count to avoid noisy sigmas.
            n = int(count_by_pid.get(pid, 1))
            alpha = float(n / (n + 8.0))
            sigma = alpha * sigma + (1.0 - alpha) * sigma_global
            patient_sigma[pid] = float(np.clip(sigma, 70.0, 1000.0))
            patient_n[pid] = n

    # Prepare test raw features (not normalized) for prediction
    TEST_RAW = pd.read_csv(os.path.join(BASE_INPUT, "test.csv"))
    TEST_RAW = create_typical_fvc(TEST_RAW)
    TEST_RAW = add_onehots(TEST_RAW)

    test_base_fvc = TEST_RAW.set_index("Patient")["FVC"].to_dict()
    test_base_week = TEST_RAW.set_index("Patient")["Weeks"].to_dict()

    test_feat_map = {}
    for _, r in TEST_RAW.iterrows():
        pid = r["Patient"]
        test_feat_map[pid] = {
            "typical_fvc": float(r["typical_fvc"]),
            "Age": float(r["Age"]),
            "Male": float(r["Male"]),
            "Female": float(r["Female"]),
            "Never smoked": float(r["Never smoked"]),
            "Currently smokes": float(r["Currently smokes"]),
            "Ex-smoker": float(r["Ex-smoker"]),
        }

    pw = SAMPLE_SUBMISSION["Patient_Week"].values
    preds_fvc = np.zeros(len(SAMPLE_SUBMISSION), dtype=np.float64)
    preds_conf = np.zeros(len(SAMPLE_SUBMISSION), dtype=np.float64)

    for i, s in enumerate(pw):
        pid, wk = s.split("_")
        wk = float(wk)

        y0 = float(test_base_fvc.get(pid, float(TRAIN_DF2["FVC"].mean())))
        w0 = float(test_base_week.get(pid, 0.0))
        dw = wk - w0

        if beta is not None and pid in test_feat_map:
            f = test_feat_map[pid]
            x = np.array(
                [
                    dw,
                    f["typical_fvc"],
                    f["Age"],
                    f["Male"],
                    f["Female"],
                    f["Never smoked"],
                    f["Currently smokes"],
                    f["Ex-smoker"],
                ],
                dtype=np.float64,
            )
            x1 = np.concatenate([np.ones((1,), dtype=np.float64), x], axis=0)
            dy = float(x1 @ beta)
        else:
            dy = 0.0

        fvc = y0 + dy
        preds_fvc[i] = float(np.clip(fvc, 0.0, 10000.0))

        # Change (metric-aligned): confidence grows mildly with |dw|, preventing overconfidence far from baseline.
        base_sigma = float(patient_sigma.get(pid, sigma_global))
        conf = base_sigma * (1.0 + 0.004 * min(abs(dw), 250.0))
        preds_conf[i] = float(np.clip(conf, 70.0, 1000.0))

    predictions = np.zeros((len(SAMPLE_SUBMISSION), 3), dtype="float32")
    predictions[:, 1] = preds_fvc.astype("float32")
    predictions[:, 2] = preds_conf.astype("float32")
    predictions[:, 0] = 0.0




In [9]:
def denormalize(y):
    return y * (MIN_MAX["FVC"][1] - MIN_MAX["FVC"][0]) + MIN_MAX["FVC"][0]


if len(models) > 0:
    predictions = predictions / len(models)
    predictions = denormalize(predictions)
    FVC = predictions[:, 1]
    Confidence = predictions[:, 2] - predictions[:, 0]
else:
    FVC = predictions[:, 1]
    Confidence = predictions[:, 2]

FVC = np.asarray(FVC).astype(np.float64)
Confidence = np.asarray(Confidence).astype(np.float64)
Confidence = np.maximum(Confidence, 70.0)

SAMPLE_SUBMISSION["FVC"] = np.round(FVC).astype(int)
SAMPLE_SUBMISSION["Confidence"] = np.round(Confidence).astype(int)

SAMPLE_SUBMISSION = SAMPLE_SUBMISSION[["Patient_Week", "FVC", "Confidence"]]
SAMPLE_SUBMISSION.to_csv("submission.csv", index=False)

SAMPLE_SUBMISSION.head()

,Patient_Week,FVC,Confidence
0,ID00126637202218610655908_-3,2418,225
1,ID00126637202218610655908_-2,2414,225
2,ID00126637202218610655908_-1,2411,224
3,ID00126637202218610655908_0,2407,223
4,ID00126637202218610655908_1,2404,222
